In [54]:
import pandas as pd
books = pd.read_csv('C:/Users/Admin/Downloads/archive/books.csv')
tags = pd.read_csv('C:/Users/Admin/Downloads/archive/tags.csv')
book_tags = pd.read_csv('C:/Users/Admin/Downloads/archive/book_tags.csv')

In [3]:
books.head(3)

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...


In [37]:
sl_sach = books['book_id'].nunique()
sl_sach

10000

In [4]:
tags.head(3)

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-


In [6]:
book_tags.head(13)

,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716
5,1,11743,9954
6,1,14017,7169
7,1,5207,6221
8,1,22743,4974
9,1,32989,4364


In [11]:
book_tags.describe()

,book_id,tag_id,count
count,9.999120e+05,999912.000000,999912.000000
mean,5.263442e+06,16324.527073,208.869633
std,7.574057e+06,9647.846196,3501.265173
min,1.000000e+00,0.000000,-1.000000
25%,4.622700e+04,8067.000000,7.000000
50%,3.948410e+05,15808.000000,15.000000
75%,9.378297e+06,24997.000000,40.000000
max,3.328864e+07,34251.000000,596234.000000


In [ ]:
book_tags = book_tags.rename(columns={'goodreads_book_id': 'book_id'})

In [24]:



# ---------------------------------------------------------
# YÊU CẦU 2: Lọc ra các tag_id có số đếm "đáng kể"
# ---------------------------------------------------------

# Bước a: Tính tổng số 'count' của TỪNG quyển sách (theo book_id)
# transform('sum') sẽ tạo ra một cột tạm thời chứa tổng count của cuốn sách tương ứng với từng dòng
tong_count_tung_sach = book_tags.groupby('book_id')['count'].transform('sum')

# Bước b: Đặt ra một ngưỡng (threshold) để đánh giá mức độ "đáng kể"
# Ví dụ ở đây: Số count của tag_id đó phải chiếm ít nhất 5% (0.05) tổng số count của quyển sách
nguong_phan_tram = 0.02

# Bước c: Lọc DataFrame
df_filtered = book_tags[book_tags['count'] >= (tong_count_tung_sach * nguong_phan_tram)]

# (Tùy chọn) Reset lại index sau khi lọc cho gọn gàng
df_filtered = df_filtered.reset_index(drop=True)

# Xem kết quả
df_filtered.describe()

,book_id,tag_id,count
count,5.341800e+04,53418.000000,53418.000000
mean,5.024860e+06,18434.226422,3209.922872
std,7.356218e+06,9589.651018,14821.335043
min,1.000000e+00,71.000000,7.000000
25%,4.418600e+04,9124.000000,174.000000
50%,3.491915e+05,15359.000000,402.000000
75%,8.533018e+06,30358.000000,1217.750000
max,3.328864e+07,34249.000000,596234.000000


In [25]:
df_filtered.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/test.ipynbbook_tags_filtered.csv', index=False)

In [26]:
# Đếm số lượng giá trị khác nhau trong cột 'book_id'
so_luong_sach = book_tags['book_id'].nunique()
so_luong_sach_1 = df_filtered['book_id'].nunique()
print(f"Có tổng cộng {so_luong_sach} đầu sách khác nhau trong dữ liệu.")
print(f"Có tổng cộng {so_luong_sach_1} đầu sách khác nhau sau khi lọc.")

Có tổng cộng 10000 đầu sách khác nhau trong dữ liệu.
Có tổng cộng 10000 đầu sách khác nhau sau khi lọc.


In [27]:
book_tags_filtered=pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/test.ipynbbook_tags_filtered.csv')
book_tags_filtered.head(3)

,book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173


In [28]:
# Thực hiện ghép (merge) bảng book_tags_filtered với bảng tags
# on='tag_id': chỉ định cột chung để so khớp
# how='left': giữ lại toàn bộ dữ liệu ở bảng bên trái (book_tags_filtered) 
#             và chỉ lấy thêm tên từ bảng bên phải (tags)
book_tags_new = pd.merge(book_tags_filtered, tags, on='tag_id', how='left')

# Kiểm tra kết quả
book_tags_new.head()

,book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy
2,1,11557,34173,favorites
3,1,8717,12986,currently-reading
4,1,33114,12716,young-adult


In [29]:
book_tags_new.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_tags.csv', index=False)

In [32]:
# Đếm xem mỗi tag_name xuất hiện trên bao nhiêu cuốn sách khác nhau
tag_phu_song_rong = book_tags_new.groupby('tag_name')['book_id'].count().reset_index()

# Đổi tên cột cho dễ hiểu và sắp xếp
tag_phu_song_rong = tag_phu_song_rong.rename(columns={'book_id': 'so_luong_sach'})
tag_phu_song_rong = tag_phu_song_rong.sort_values(by='so_luong_sach', ascending=False).reset_index(drop=True)

print("--- TOP 10 TAGS XUẤT HIỆN TRÊN NHIỀU SÁCH NHẤT ---")
print(tag_phu_song_rong.head(20))

--- TOP 10 TAGS XUẤT HIỆN TRÊN NHIỀU SÁCH NHẤT ---
              tag_name  so_luong_sach
0              to-read           9803
1    currently-reading           5810
2              fiction           3907
3            favorites           3301
4              fantasy           1898
5              romance           1269
6          young-adult           1249
7              mystery           1102
8          non-fiction            904
9                owned            848
10         books-i-own            799
11              series            776
12            classics            757
13  historical-fiction            744
14                  ya            667
15            thriller            575
16          paranormal            534
17     science-fiction            531
18           childrens            495
19              sci-fi            482


In [34]:
books_tags=pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_tags.csv')

# Thực hiện gom nhóm theo book_id và gộp tag_name thành list
book_tags_list = books_tags.groupby('book_id')['tag_name'].apply(list).reset_index()

# Đổi tên cột cho rõ nghĩa (tùy chọn)
book_tags_list = book_tags_list.rename(columns={'tag_name': 'tag_list'})

# Kiểm tra kết quả
book_tags_list.head(10)

,book_id,tag_list
0,1,"[to-read, fantasy, favorites, currently-readin..."
1,2,"[to-read, currently-reading, fantasy, favorite..."
2,3,"[to-read, favorites, fantasy, currently-reading]"
3,5,"[favorites, fantasy, currently-reading, young-..."
4,6,"[fantasy, young-adult, fiction, harry-potter, ..."
5,8,"[to-read, favorites, fantasy]"
6,10,"[to-read, favorites, fantasy, currently-reading]"
7,11,"[to-read, currently-reading, science-fiction, ..."
8,13,"[to-read, currently-reading, favorites, scienc..."
9,21,"[to-read, currently-reading, history, nonficti..."


In [38]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='object')

In [56]:
# Điền các giá trị thiếu của original_title bằng giá trị của title ở cùng dòng
books['original_title'] = books['original_title'].fillna(books['title'])

In [58]:
columns = ['id','best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13','title', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count']
books=books.drop(columns, axis=1)
books.head(10)

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...
5,11870085,John Green,2012.0,The Fault in Our Stars,eng,47994,92723,327550,698471,1311871,https://images.gr-assets.com/books/1360206420m...,https://images.gr-assets.com/books/1360206420s...
6,5907,J.R.R. Tolkien,1937.0,The Hobbit or There and Back Again,en-US,46023,76784,288649,665635,1119718,https://images.gr-assets.com/books/1372847500m...,https://images.gr-assets.com/books/1372847500s...
7,5107,J.D. Salinger,1951.0,The Catcher in the Rye,eng,109383,185520,455042,661516,709176,https://images.gr-assets.com/books/1398034300m...,https://images.gr-assets.com/books/1398034300s...
8,960,Dan Brown,2000.0,Angels & Demons,en-CA,77841,145740,458429,716569,680175,https://images.gr-assets.com/books/1303390735m...,https://images.gr-assets.com/books/1303390735s...
9,1885,Jane Austen,1813.0,Pride and Prejudice,eng,54700,86485,284852,609755,1155673,https://images.gr-assets.com/books/1320399351m...,https://images.gr-assets.com/books/1320399351s...


In [59]:
# Gộp tag_list từ book_tags_list vào bảng books
# on='book_id': Khớp lệnh dựa trên mã sách
# how='left': Giữ lại tất cả sách trong bảng books, 
#             nếu sách nào không có tag thì giá trị sẽ là NaN
bookss = pd.merge(books, book_tags_list, on='book_id', how='left')
bookss.head()

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tag_list
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,"[favorites, currently-reading, young-adult, fi..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"[to-read, favorites, fantasy, currently-reading]"
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"[young-adult, fantasy, favorites, vampires, ya..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,"[classics, favorites, to-read, classic, histor..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"[classics, favorites, fiction, classic, books-..."


In [60]:


# Xử lý các dòng không có tag (NaN) thành một danh sách trống []
# Điều này giúp code không bị lỗi khi bạn thực hiện các thao tác xử lý chuỗi/list sau này
bookss['tag_list'] = bookss['tag_list'].apply(lambda d: d if isinstance(d, list) else [])

# Kiểm tra 5 dòng đầu tiên
bookss.head()

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tag_list
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,"[favorites, currently-reading, young-adult, fi..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"[to-read, favorites, fantasy, currently-reading]"
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"[young-adult, fantasy, favorites, vampires, ya..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,"[classics, favorites, to-read, classic, histor..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"[classics, favorites, fiction, classic, books-..."


In [65]:
bookss = bookss.rename(columns={'tag_list': 'tags'})
bookss.head()

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tags
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,"[favorites, currently-reading, young-adult, fi..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"[to-read, favorites, fantasy, currently-reading]"
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"[young-adult, fantasy, favorites, vampires, ya..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,"[classics, favorites, to-read, classic, histor..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"[classics, favorites, fiction, classic, books-..."


In [67]:
bookss.columns  


Index(['book_id', 'authors', 'original_publication_year', 'original_title',
       'language_code', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4',
       'ratings_5', 'image_url', 'small_image_url', 'tags'],
      dtype='object')

In [66]:
# Đếm số lượng giá trị null trong mỗi cột
so_luong_null = bookss.isna().sum()

print(so_luong_null)

book_id                         0
authors                         0
original_publication_year      21
original_title                  0
language_code                1084
ratings_1                       0
ratings_2                       0
ratings_3                       0
ratings_4                       0
ratings_5                       0
image_url                       0
small_image_url                 0
tags                            0
dtype: int64


In [68]:
bookss= bookss[['book_id', 'authors', 'original_publication_year', 'original_title',
       'language_code', 'tags', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4',
       'ratings_5', 'image_url', 'small_image_url']]

In [79]:
bookss.describe()

,book_id,original_publication_year,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5
count,1.000000e+04,9979.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,1.000000e+04
mean,5.264697e+06,1981.987674,1345.040600,3110.885000,11475.893800,1.996570e+04,2.378981e+04
std,7.575462e+06,152.576665,6635.626263,9717.123578,28546.449183,5.144736e+04,7.976889e+04
min,1.000000e+00,-1750.000000,11.000000,30.000000,323.000000,7.500000e+02,7.540000e+02
25%,4.627575e+04,1990.000000,196.000000,656.000000,3112.000000,5.405750e+03,5.334000e+03
50%,3.949655e+05,2004.000000,391.000000,1163.000000,4894.000000,8.269500e+03,8.836000e+03
75%,9.382225e+06,2011.000000,885.000000,2353.250000,9287.000000,1.602350e+04,1.730450e+04
max,3.328864e+07,2017.000000,456191.000000,436802.000000,793319.000000,1.481305e+06,3.011543e+06


In [69]:
bookss.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags.csv', index=False)

In [7]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import uuid

# ==========================================
# PHẦN 1: CẤU HÌNH THÔNG TIN (BẠN CẦN THAY ĐỔI)
# ==========================================

# 1. Đường dẫn đến file CSV
DUONG_DAN_FILE = r'E:\CodeFolder\Github_project\BookRecProject\Backend\books_with_tags.csv'

# 2. Tên Server SQL của bạn 
TEN_SERVER = r'LEHIEU\SQLEXPRESS' 
TEN_DATABASE = 'BookRecDb'

# 4. Tên bảng trong Database
TEN_BANG = 'Books' 
TEN_SCHEMA = 'dbo'

# 5. Cấu hình Driver SQL Server
DRIVER = 'ODBC Driver 17 for SQL Server' 


# ==========================================
# PHẦN 2: CHUỖI KẾT NỐI (CHỌN 1 TRONG 2 CÁCH)
# ==========================================

# CÁCH 1: Dùng Windows Authentication (Đăng nhập không cần pass) - KHUYÊN DÙNG
params = urllib.parse.quote_plus(
    f"DRIVER={DRIVER};"
    f"SERVER={TEN_SERVER};"
    f"DATABASE={TEN_DATABASE};"
    f"Trusted_Connection=yes;"
)


# ==========================================
# PHẦN 3: THỰC THI (KHÔNG CẦN SỬA)
# ==========================================

def run_import():
    print(f"1. Đang đọc file CSV từ: {DUONG_DAN_FILE}...")
    try:
        # on_bad_lines='skip': Bỏ qua các dòng bị lỗi lệch cột do dấu phẩy
        df = pd.read_csv(DUONG_DAN_FILE, on_bad_lines='skip')
    except Exception as e:
        print(f"❌ LỖI ĐỌC FILE CSV: {e}")
        return

    # Kiểm tra xem file có dữ liệu không
    if df.empty:
        print("❌ LỖI: DataFrame rỗng! Hãy kiểm tra lại đường dẫn file hoặc cấu trúc file CSV.")
        return
        
    so_dong = len(df)
    print(f"✅ Đã đọc thành công {so_dong} dòng dữ liệu.")
    
    # ---------------------------------------------------------
    # XỬ LÝ DỮ LIỆU: CHUYỂN UUID VÀ XÓA LỖI NULL
    # ---------------------------------------------------------
    print("-> Đang xử lý dữ liệu (Chuyển UUID và điền dữ liệu trống)...")
    try:
        # 1. Chuyển đổi book_id sang chuẩn UUID của SQL Server
        df['book_id'] = df['book_id'].apply(lambda x: str(uuid.UUID(int=int(x))))
        
        # 2. Xử lý các cột bị trống (NULL) để tránh lỗi Database từ chối
        df['language_code'] = df['language_code'].fillna('unknown')
        df['original_title'] = df['original_title'].fillna('Unknown Title')
        df['authors'] = df['authors'].fillna('Unknown Author')
        df['tags'] = df['tags'].fillna('[]')
        df['image_url'] = df['image_url'].fillna('')
        df['small_image_url'] = df['small_image_url'].fillna('')
        df['original_publication_year'] = df['original_publication_year'].fillna(0)

    except Exception as e:
        print(f"❌ LỖI XỬ LÝ DỮ LIỆU: {e}")
        return
    # ---------------------------------------------------------

    print("\n--- Mẫu dữ liệu (Dòng 1) sau khi xử lý ---")
    print(df.iloc[0])
    print("------------------------------------------\n")

    print("2. Đang thiết lập kết nối đến SQL Server...")
    try:
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    except Exception as e:
        print(f"❌ LỖI KẾT NỐI: {e}")
        return

    print("3. Đang đẩy dữ liệu vào Database. Vui lòng chờ...")
    try:
        # Đẩy dữ liệu. if_exists='append' nghĩa là chèn thêm vào bảng đã có.
        df.to_sql(TEN_BANG, con=engine, if_exists='append', index=False, schema=TEN_SCHEMA)
        
        # Xác minh lại dữ liệu đã thực sự vào DB chưa
        with engine.connect() as conn:
            result = conn.execute(f"SELECT COUNT(*) FROM {TEN_SCHEMA}.{TEN_BANG}")
            count_db = result.scalar()
            print(f"✅ QUÁ TRÌNH IMPORT HOÀN TẤT THÀNH CÔNG!")
            print(f"📊 Số dòng hiện tại trong bảng {TEN_SCHEMA}.{TEN_BANG} là: {count_db}")
            
    except Exception as e:
        print(f"❌ LỖI KHI PUSH VÀO DATABASE: {e}")

# Chạy hàm
if __name__ == "__main__":
    run_import()

1. Đang đọc file CSV từ: E:\CodeFolder\Github_project\BookRecProject\Backend\books_with_tags.csv...
✅ Đã đọc thành công 10000 dòng dữ liệu.
-> Đang xử lý dữ liệu (Chuyển UUID và điền dữ liệu trống)...

--- Mẫu dữ liệu (Dòng 1) sau khi xử lý ---
book_id                                   00000000-0000-0000-0000-0000002a38cc
authors                                                        Suzanne Collins
original_publication_year                                               2008.0
original_title                                                The Hunger Games
language_code                                                              eng
tags                         ['favorites', 'currently-reading', 'young-adul...
ratings_1                                                                66715
ratings_2                                                               127936
ratings_3                                                               560092
ratings_4                                   

In [4]:
import pandas as pd
import numpy as np
books_with_tags = pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags.csv')


In [5]:
books_with_tags['cost'] = np.random.randint(50, 101, size=len(books_with_tags))
books_with_tags.head()

,book_id,authors,original_publication_year,original_title,language_code,tags,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,cost
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,"['favorites', 'currently-reading', 'young-adul...",66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,70
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,"['to-read', 'favorites', 'fantasy', 'currently...",75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,59
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,"['young-adult', 'fantasy', 'favorites', 'vampi...",456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,93
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,"['classics', 'favorites', 'to-read', 'classic'...",60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,72
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,"['classics', 'favorites', 'fiction', 'classic'...",86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,86


In [6]:
books_with_tags.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags.csv', index=False)